# Sepsis Early Risk Classification

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `sepsis`

## 2 · Project Overview

We predict **early sepsis onset** in ICU patients using vital signs (heart rate, respiratory rate, temperature, blood pressure), lab values (WBC, lactate, creatinine), and patient history.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given an ICU patient's vital signs, lab results, age, ICU stay duration, and infection history, predict whether they will develop sepsis.

## 5 · Why This Project Matters

- **Sepsis kills ~270,000 Americans annually** — early detection saves lives.
- Every hour of delayed treatment increases mortality by ~4-8%.
- ML-based early warning systems can alert clinicians hours before clinical deterioration.
- Demonstrates critical-care binary classification with high-stakes imbalance.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 9,000 |
| **Features** | heart_rate, resp_rate, temperature_c, systolic_bp, wbc_count, lactate, age, icu_hours, prev_infection, creatinine |
| **Target** | sepsis (0 = no, 1 = yes) |
| **Class balance** | ~70/30 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "sepsis"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: sepsis
Save dir: E:\Github\Machine-Learning-Projects\Classification\Sepsis Early Risk Classification


## 11 · Dataset Download or Loading

Synthetic dataset: 9,000 ICU patient records with vital signs and sepsis outcome.

In [4]:
np.random.seed(SEED)
n = 9000
heart_rate = np.round(np.random.normal(85, 15, n).clip(40, 180), 0).astype(int)
resp_rate = np.round(np.random.normal(18, 4, n).clip(8, 45), 0).astype(int)
temperature_c = np.round(np.random.normal(37.0, 0.8, n).clip(34, 42), 1)
systolic_bp = np.round(np.random.normal(120, 20, n).clip(60, 200), 0).astype(int)
wbc_count = np.round(np.random.lognormal(2.2, 0.4, n).clip(1, 40), 1)
lactate = np.round(np.random.lognormal(0.3, 0.5, n).clip(0.3, 15), 1)
age = np.random.randint(18, 90, n)
icu_hours = np.random.randint(1, 168, n)
prev_infection = np.random.binomial(1, 0.2, n)
creatinine = np.round(np.random.lognormal(0.1, 0.4, n).clip(0.3, 8), 1)

score = (0.02 * heart_rate + 0.05 * resp_rate + 0.3 * (temperature_c - 37)
         + (-0.01) * systolic_bp + 0.1 * wbc_count + 0.4 * lactate
         + 0.01 * age + 0.005 * icu_hours + 0.5 * prev_infection
         + 0.2 * creatinine + np.random.normal(0, 1.2, n))
sepsis = (score > np.percentile(score, 70)).astype(int)

df = pd.DataFrame({"heart_rate": heart_rate, "resp_rate": resp_rate,
                    "temperature_c": temperature_c, "systolic_bp": systolic_bp,
                    "wbc_count": wbc_count, "lactate": lactate, "age": age,
                    "icu_hours": icu_hours, "prev_infection": prev_infection,
                    "creatinine": creatinine, "sepsis": sepsis})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['sepsis'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (9000, 11)
Class balance:
sepsis
0    0.7
1    0.3
Name: proportion, dtype: float64


,heart_rate,resp_rate,temperature_c,systolic_bp,wbc_count,lactate,age,icu_hours,prev_infection,creatinine,sepsis
0,92,21,37.3,104,11.3,2.3,53,94,1,0.7,1
1,83,22,36.4,143,8.5,0.9,33,140,1,0.5,1
2,95,20,37.6,136,10.6,2.5,77,100,0,3.7,1
3,108,10,36.5,120,10.4,1.3,69,102,0,0.9,0
4,81,17,37.6,123,7.4,2.1,39,27,0,0.7,1


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (9000, 11)

Missing values:
heart_rate        0
resp_rate         0
temperature_c     0
systolic_bp       0
wbc_count         0
lactate           0
age               0
icu_hours         0
prev_infection    0
creatinine        0
sepsis            0
dtype: int64

Duplicate rows: 0

Dtypes:
heart_rate          int64
resp_rate           int64
temperature_c     float64
systolic_bp         int64
wbc_count         float64
lactate           float64
age                 int32
icu_hours           int32
prev_infection      int32
creatinine        float64
sepsis              int64
dtype: object

Target 'sepsis' confirmed. Value counts:
sepsis
0    6300
1    2700
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["heart_rate", "resp_rate", "temperature_c", "wbc_count", "lactate", "creatinine"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
plt.suptitle("Vital Signs by Sepsis Status", fontsize=14)
plt.tight_layout()
plt.show()

corr = df.select_dtypes(include="number").corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `sepsis`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "salmon"], edgecolor="black")
ax.set_title("Sepsis Distribution")
ax.set_xticklabels(["No Sepsis (0)", "Sepsis (1)"], rotation=0)
plt.tight_layout()
plt.show()
print(f"Sepsis rate: {df[TARGET].mean():.1%}")

Sepsis rate: 30.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (7200, 10), Test: (1800, 10)
Train class distribution:
sepsis
0    0.7
1    0.3
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: None (all numeric features).
- **Scaling**: Not needed for tree models.
- **Class balance**: ~30% sepsis.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["sirs_score"] = ((X_train["heart_rate"] > 90).astype(int)
                         + (X_train["resp_rate"] > 20).astype(int)
                         + (X_train["temperature_c"] > 38).astype(int)
                         + (X_train["wbc_count"] > 12).astype(int))
X_test["sirs_score"] = ((X_test["heart_rate"] > 90).astype(int)
                        + (X_test["resp_rate"] > 20).astype(int)
                        + (X_test["temperature_c"] > 38).astype(int)
                        + (X_test["wbc_count"] > 12).astype(int))

X_train["shock_index"] = X_train["heart_rate"] / (X_train["systolic_bp"] + 1)
X_test["shock_index"] = X_test["heart_rate"] / (X_test["systolic_bp"] + 1)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (12): ['heart_rate', 'resp_rate', 'temperature_c', 'systolic_bp', 'wbc_count', 'lactate', 'age', 'icu_hours', 'prev_infection', 'creatinine', 'sirs_score', 'shock_index']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.7611
F1       : 0.4881

              precision    recall  f1-score   support

           0       0.78      0.92      0.84      1260
           1       0.68      0.38      0.49       540

    accuracy                           0.76      1800
   macro avg       0.73      0.65      0.67      1800
weighted avg       0.75      0.76      0.74      1800



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
NearestCentroid                0.693889           0.688228  0.752234  0.704580   0.731650  0.693889    0.026132
Perceptron                     0.679444           0.672619  0.734184  0.690821   0.719138  0.679444    0.038585
GaussianNB                     0.739444           0.661508  0.746454  0.730424   0.727120  0.739444    0.021707
LogisticRegression             0.760556           0.652249  0.765169  0.737189   0.747781  0.760556    0.033767
LinearDiscriminantAnalysis     0.758333           0.650132  0.765417  0.735036   0.744783  0.758333    0.032047
SGDClassifier                  0.746111           0.649868  0.751080  0.729060   0.729718  0.746111    0.068772
CalibratedClassifierCV         0.758889           0.649471  0.765291  

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.7194
F1: 0.4701


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.4752  (1.5s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[97]	valid_0's binary_logloss: 0.5181
LightGBM F1: 0.4871  (1.0s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression    0.7611  0.4881     0.6833  0.3796
LightGBM               0.7578  0.4871     0.6677  0.3833
CatBoost               0.7594  0.4752     0.6877  0.3630
FLAML                  0.7194  0.4701     0.5424  0.4148

Top 3 models: ['Logistic Regression', 'LightGBM', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  Logistic Regression:
    Accuracy : 0.7611
    F1       : 0.4881
    Precision: 0.6833
    Recall   : 0.3796

  LightGBM:
    Accuracy : 0.7578
    F1       : 0.4871
    Precision: 0.6677
    Recall   : 0.3833

  CatBoost:
    Accuracy : 0.7594
    F1       : 0.4752
    Precision: 0.6877
    Recall   : 0.3630


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.92      0.84      1260
           1       0.68      0.38      0.49       540

    accuracy                           0.76      1800
   macro avg       0.73      0.65      0.67      1800
weighted avg       0.75      0.76      0.74      1800


Total errors: 430 / 1800 (23.9%)

Sample misclassifications:
      heart_rate  resp_rate  temperature_c  systolic_bp  wbc_count  lactate  age  icu_hours  prev_infection  creatinine  sirs_score  shock_index  actual  predicted  correct
8163          85         15           36.2          128        8.4      1.1   57         51               1         1.2           0     0.658915       1          0    False
3464          88         19           36.9          119       10.7      1.2   71         66               0         1.1           0     0.733333       1          0    False
8263          84         21         

## 25 · Interpretation and Business Insight

**Key findings:**
- **Lactate** is the strongest sepsis predictor (tissue hypoxia marker).
- **Previous infection** greatly increases sepsis risk.
- **SIRS score** (engineered) effectively combines vital sign abnormalities.
- **Shock index** (HR/SBP) indicates hemodynamic instability.
- **WBC count** and **creatinine** signal immune response and organ dysfunction.

**Business takeaway:** Monitor lactate and SIRS criteria closely. Alert on shock index > 0.7.

## 26 · Limitations

1. Synthetic data — real sepsis has complex temporal dynamics.
2. No temporal sequence (vital signs over time).
3. Missing blood culture results and antibiotic timing.
4. No SOFA/qSOFA score components fully modeled.
5. Binary outcome ignores sepsis severity stages.

## 27 · How to Improve This Project

1. Use MIMIC-III/IV real ICU data.
2. Add temporal sequences (hourly vitals).
3. Include blood culture and antibiotic data.
4. Model time-to-sepsis (survival analysis).
5. Add SOFA sub-scores for organ dysfunction.

## 28 · Production Considerations

- Deploy as a real-time ICU early warning system.
- Optimize for **high recall** (missing sepsis is fatal).
- Alert clinicians with probability score and contributing factors.
- Must undergo clinical validation (FDA approval).
- Monitor for alert fatigue and false positive rates.

## 29 · Common Mistakes

1. Optimizing for accuracy instead of recall.
2. Not considering the temporal nature of vital signs.
3. Deploying without clinical validation.
4. Ignoring alert fatigue from too many false positives.
5. Not accounting for treatment effects in the data.

## 30 · Mini Challenge / Exercises

1. Set threshold for 95% recall — what is the false positive rate?
2. Remove `lactate` — how much does performance drop?
3. Plot SIRS score distribution by sepsis status.
4. Try time-aware splitting (by `icu_hours`).
5. Calculate the cost of false negatives vs. false positives in ICU context.

## 31 · Final Summary / Key Takeaways

1. **Lactate** and **previous infection** are the strongest sepsis predictors.
2. **SIRS score** (engineered) effectively summarizes vital sign abnormalities.
3. **Shock index** detects hemodynamic compromise.
4. **High recall** is critical — missing sepsis cases is unacceptable.
5. **Temporal data** would dramatically improve real-world performance.
6. **Clinical validation** is mandatory before deployment.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Sepsis Early Risk Classification\metrics.json
{
  "CatBoost": {
    "accuracy": 0.7594,
    "f1": 0.4752,
    "precision": 0.6877,
    "recall": 0.363
  },
  "LightGBM": {
    "accuracy": 0.7578,
    "f1": 0.4871,
    "precision": 0.6677,
    "recall": 0.3833
  },
  "Logistic Regression": {
    "accuracy": 0.7611,
    "f1": 0.4881,
    "precision": 0.6833,
    "recall": 0.3796
  },
  "FLAML": {
    "accuracy": 0.7194,
    "f1": 0.4701,
    "precision": 0.5424,
    "recall": 0.4148
  }
}
